# Runde 1b · L-CAL (Konto A, CPU-Laufzeit mit hohem RAM)

Rechnet `l_cal.json` mit dem **eingefrorenen** Code (`r1b_l_cal.py`, `r1b_l_stats.py`, `r1b_l_rules.py` am Freeze-Commit; Plan 9.1).
Keine GPU nötig: *Laufzeit → Laufzeittyp ändern → CPU*, **hoher RAM** an. Zwischenstände liegen in Drive
(`dsse_runs/r1b_lcal/cache`), ein Abbruch verliert also nur das laufende Teilstück: einfach erneut „Alle ausführen“.
Ergebnis: `dsse_runs/r1b_lcal/l_cal.json` (danach dem Koordinator melden; er committet die Datei).

In [ ]:
import os, sys, subprocess, json, glob, time, hashlib
from google.colab import drive; drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/dsse_runs/r1b_lcal"; CACHE = f"{OUT}/cache"; os.makedirs(CACHE, exist_ok=True)
REPO = "/content/DSSE"; FREEZE = "4de7a3b41169704dd811e11965727e0e64d6b8f9"
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "-q", "-b", "ccr-6fe02b21-est040", "https://github.com/Erikiss/DSSE", REPO], check=True)
R = f"{REPO}/analysis/ju_space/round1b"
d = subprocess.run(["git", "-C", REPO, "diff", "--stat", FREEZE, "HEAD", "--", f"{R}/r1b_l_cal.py", f"{R}/r1b_l_stats.py", f"{R}/r1b_l_rules.py"],
                   capture_output=True, text=True).stdout
assert not d.strip(), "L-CAL-Code weicht vom Freeze-Commit ab: " + d
print("Code = Freeze-Stand; Kerne:", os.cpu_count())

In [ ]:
# Teilstücke parallel (je 2 Threads): (a) nach Profilen, dann die übrigen Punkte; jedes fertige Teilstück liegt sofort in Drive
PART = f'''
import os, sys, pickle
sys.path.insert(0, "{R}")
import r1b_l_cal as LC
prof, out = sys.argv[1], sys.argv[2]
os.environ["R1B_LCAL_A_PROFILES"] = prof
r = LC.item_a(False, 5); pickle.dump(r, open(out + ".tmp", "wb")); os.replace(out + ".tmp", out)
print("fertig", prof, r["hits"], flush=True)
'''
open("/content/a_part.py", "w").write(PART)
ncpu = os.cpu_count() or 2
jobs = []
for prof in ("localist", "redundant", "dense", "dthr", "xor3"):
    p = f"{CACHE}/a_{prof}.pkl"
    if not os.path.exists(p):
        jobs.append(["python3", "/content/a_part.py", prof, p])
WRAP = f"{REPO}/results/ju_space/plans/runde1b/arbeitsstand/lcal/lcal_resumable.py"
rest = "".join(k for k in "bcdefghijklmn" if not os.path.exists(f"{CACHE}/{k}.pkl"))
if rest:
    jobs.append(["python3", WRAP, f"{OUT}/l_cal.json", CACHE, "--items", rest])
thr = str(max(1, ncpu // max(1, min(len(jobs), ncpu))))
procs = []
for j in jobs:
    procs.append(subprocess.Popen(j, env=dict(os.environ, OMP_NUM_THREADS=thr), stdout=open(f"/content/log_{len(procs)}.txt", "w"), stderr=subprocess.STDOUT))
    while sum(p.poll() is None for p in procs) >= max(1, ncpu // int(thr)):
        time.sleep(30)
while any(p.poll() is None for p in procs):
    time.sleep(60); print(time.strftime("%H:%M"), "fertig in Drive:", sorted(os.path.basename(x) for x in glob.glob(f"{CACHE}/*.pkl")), flush=True)
print("Teilstücke fertig.")

In [ ]:
# (a) zusammenführen und l_cal.json schreiben (Torprüfung G_cal_L1..L5 durch r1b_l_cal.run)
import pickle
sys.path.insert(0, R); import r1b_l_cal as LC
parts = [pickle.load(open(f"{CACHE}/a_{p}.pkl", "rb")) for p in ("localist", "redundant", "dense", "dthr", "xor3")]
hits = {k: v for x in parts for k, v in x["hits"].items()}; need = 4
a = dict(hits=hits, seeds=5, trace=sum((x["trace"] for x in parts), []), pass_=all(hits[p] >= need for p in hits))
pickle.dump(a, open(f"{CACHE}/a.pkl", "wb"))
subprocess.run(["python3", WRAP, f"{OUT}/l_cal.json", CACHE], check=True)
print(json.dumps(json.load(open(f"{OUT}/l_cal.json"))["gates"], indent=1))